---
## 1. 알고리즘 — 정렬

In [ ]:
# ── 기본 정렬 (라이브러리 사용) ──────────────────────
arr = [3, 1, 4, 1, 5, 9, 2, 6]

arr.sort()                          # 오름차순 (제자리 정렬)
arr.sort(reverse=True)              # 내림차순
sorted(arr)                         # 새 리스트 반환
sorted(arr, key=lambda x: x[1])    # 기준값 지정 (2번째 요소 기준)

# ── 딕셔너리 활용 ──────────────────────────────────
d = {}
for item in arr:
    d[item] = d.get(item, 0) + 1   # 빈도 카운트

# ── 리스트 컴프리헨션 ───────────────────────────────
result = [x * 2 for x in arr if x > 0]

In [ ]:
# ── 버블 정렬 (라이브러리 없이) ──────────────────────
# 개념: 옆에 있는 두 수를 비교해서 큰 걸 오른쪽으로 밀어냄
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(n - i - 1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

print(bubble_sort([3, 1, 4, 1, 5]))  # [1, 1, 3, 4, 5]

In [ ]:
# ── 선택 정렬 (라이브러리 없이) ──────────────────────
# 개념: 전체에서 제일 작은 걸 찾아서 맨 앞으로 가져옴. 반복.
def selection_sort(arr):
    n = len(arr)
    for i in range(n):
        min_idx = i
        for j in range(i+1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
    return arr

print(selection_sort([3, 1, 4, 1, 5]))  # [1, 1, 3, 4, 5]

---
## 2. 회귀분석 (sklearn)
**흐름:** 데이터 준비 → 전처리 → train/test 분리 → 학습 → 예측 → 평가

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# ── 데이터 준비 ────────────────────────────────────
df = pd.read_csv('data.csv')    # 데이터 로드
df.head()                       # 상위 5행 확인
df.info()                       # 컬럼 타입 확인
df.isnull().sum()               # 결측치 확인

# ── 전처리 ────────────────────────────────────────
df.fillna(0, inplace=True)      # 결측치 0으로 채우기
# df.dropna(inplace=True)       # 또는 결측치 행 제거

# ── 입력/출력 분리 ─────────────────────────────────
X = df[['feature1', 'feature2']]  # 입력변수
y = df['target']                  # 예측할 값

# ── train/test 분리 (8:2) ──────────────────────────
# random_state: 결과 재현을 위한 시드값 (숫자는 무관)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── 정규화 (필요시) ────────────────────────────────
# fit_transform: train 기준으로 평균/표준편차 학습 후 변환
# transform: 학습한 값으로만 변환 (test에는 fit 금지 → data leakage 방지)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ── 모델 학습 ─────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)       # 학습 (기울기/절편 찾기)
y_pred = model.predict(X_test)    # 예측

# ── 평가 ──────────────────────────────────────────
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'MSE:  {mse:.4f}')         # 오차 제곱 평균 (낮을수록 좋음)
print(f'RMSE: {rmse:.4f}')        # MSE의 제곱근, 단위 동일 (낮을수록 좋음)
print(f'R²:   {r2:.4f}')          # 설명력 0~1 (1에 가까울수록 좋음)

---
## 3. 이미지 분류 (keras)
**흐름:** 데이터 로드 → 정규화 → 원핫인코딩 → 모델 정의 → 학습 → 평가

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# ── 데이터 로드 예시 (MNIST 기준) ──────────────────
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# ── 전처리 ────────────────────────────────────────
# 픽셀값 0~255 → 0~1 정규화
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# CNN 입력 형태 맞추기: (샘플수, 높이, 너비, 채널수)
X_train = X_train.reshape(-1, 28, 28, 1)  # 흑백이면 채널=1
X_test  = X_test.reshape(-1, 28, 28, 1)

# 레이블 원핫인코딩: 숫자 → [0,0,1,0,...] 형태
y_train = to_categorical(y_train)   # 클래스 수 자동 감지
y_test  = to_categorical(y_test)

# ── 모델 정의 ─────────────────────────────────────
# Conv2D → MaxPooling2D → Flatten → Dense 순서가 기본
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
                                    # 32: 필터수 / (3,3): 필터크기 / relu: 활성화함수
    MaxPooling2D(2, 2),             # 2x2 영역에서 최대값 추출 (크기 절반)
    Flatten(),                      # 2D → 1D로 펼치기
    Dense(64, activation='relu'),   # 은닉층
    Dense(10, activation='softmax') # 출력층: 클래스 수 = 10 (0~9)
])

# ── 컴파일 ────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # 다중분류 손실함수
    metrics=['accuracy']             # 정확도 출력
)

# ── 학습 ──────────────────────────────────────────
model.fit(
    X_train, y_train,
    epochs=5,              # 전체 데이터 반복 횟수
    batch_size=32,         # 한번에 학습할 샘플 수
    validation_split=0.2   # 학습 중 검증용 비율
)

# ── 평가 ──────────────────────────────────────────
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss:     {loss:.4f}')
print(f'Accuracy: {accuracy:.4f}')

---
## 4. 시계열 예측
**핵심:** lag feature로 변환하면 회귀 문제와 동일  
**주의:** 시간 순서가 있어서 shuffle 금지 → train_test_split 대신 직접 분리

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ── 데이터 준비 ────────────────────────────────────
df = pd.read_csv('timeseries.csv')

# 날짜 인덱스 설정
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# ── lag feature 생성 ───────────────────────────────
# 개념: 이전 값들을 입력으로 사용해서 다음 값을 예측
df['lag1'] = df['value'].shift(1)   # 1시점 전 값
df['lag2'] = df['value'].shift(2)   # 2시점 전 값
df.dropna(inplace=True)             # shift로 생긴 NaN 제거

# ── 입력/출력 분리 ─────────────────────────────────
X = df[['lag1', 'lag2']]
y = df['value']

# ── train/test 분리 ────────────────────────────────
# 시계열은 시간 순서를 지켜야 해서 인덱스로 직접 분리
# train_test_split(shuffle=True) 사용 금지!
split = int(len(df) * 0.8)         # 80% 위치
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ── 이후는 회귀와 동일 ──────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE: {rmse:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# 정확도 — 전체 중 맞은 비율
accuracy_score(y_test, y_pred)          # 0.95 → 95% 정확

# 혼동행렬 — 어떤 클래스를 헷갈렸는지
confusion_matrix(y_test, y_pred)

# 상세 리포트 — precision, recall, f1 한번에
print(classification_report(y_test, y_pred))
Precision: 맞다고 한 것 중 진짜 맞은 비율 / Recall: 실제 맞는 것 중 찾아낸 비율 / F1: precision과 recall의 조화평균

# keras버전
loss, accuracy = model.evaluate(X_test, y_test)
print(f'accuracy: {accuracy:.4f}')
